In [1]:
import os
import pandas as pd

print("=" * 60)
print("📤 EXPORTING DATA FOR POWER BI")
print("=" * 60)

# 1) Setup paths (this notebook is in /notebooks/)
base_dir = os.path.abspath("..")          # one level up from /notebooks/
clean_dir = os.path.join(base_dir, "data", "cleaned")
output_dir = os.path.join(base_dir, "output")

os.makedirs(output_dir, exist_ok=True)

print(f"\n📂 Project root: {base_dir}")
print(f"📂 Cleaned data folder: {clean_dir}")
print(f"📂 Output folder: {output_dir}")

# 2) Load cleaned data (data_analyst_jobs.csv)
clean_file = os.path.join(clean_dir, "data_analyst_jobs.csv")

if not os.path.exists(clean_file):
    raise FileNotFoundError(
        f"❌ Cleaned file not found: {clean_file}\n"
        f"Make sure your cleaning notebook saved data_analyst_jobs.csv to data/cleaned."
    )

print("\n1️⃣ Loading cleaned data...")
df = pd.read_csv(clean_file)
print(f"   ✅ Loaded {len(df)} rows")
print(f"   📊 Columns: {list(df.columns)}")

# 3) Ensure posted_date is datetime (for trends)
if "posted_date" in df.columns:
    print("\n2️⃣ Converting posted_date to datetime...")
    df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")
    print("   ✅ Date conversion done.")
else:
    print("\n2️⃣ No 'posted_date' column found. Skipping monthly trend based on dates.")

# 4) Overall metrics
print("\n3️⃣ Creating overall metrics...")

overall_metrics = pd.DataFrame({
    "Metric": ["Total Jobs", "Avg Salary", "Median Salary", "Remote %", "Hybrid %", "Onsite %"],
    "Value": [
        len(df),
        df["salary_avg"].mean(),
        df["salary_avg"].median(),
        (df["job_type"] == "Remote").sum() / len(df) * 100,
        (df["job_type"] == "Hybrid").sum() / len(df) * 100,
        (df["job_type"] == "Onsite").sum() / len(df) * 100,
    ],
})

overall_path = os.path.join(output_dir, "overall_metrics.csv")
overall_metrics.to_csv(overall_path, index=False)
print(f"   ✅ Saved: {overall_path}")

# 5) Skills summary
print("\n4️⃣ Creating skills summary...")

skill_cols = [
    "Python", "SQL", "Excel", "Power BI", "Tableau",
    "Machine Learning", "TensorFlow", "AWS", "Azure", "R",
    "JavaScript", "DAX", "Visualization"
]

# Create missing skill columns if needed
missing = [c for c in skill_cols if c not in df.columns]
if missing:
    print(f"   ⚠️ Missing skill columns found: {missing}")
    print("   ➜ Creating them from 'required_skills' column...")
    if "required_skills" not in df.columns:
        raise ValueError("❌ 'required_skills' column not found; cannot derive skill columns.")
    for skill in skill_cols:
        if skill not in df.columns:
            df[skill] = df["required_skills"].str.contains(skill, case=False, na=False).astype(int)
    print("   ✅ Skill columns created.")

skills_summary = pd.DataFrame({
    "Skill": skill_cols,
    "Job_Count": [df[skill].sum() for skill in skill_cols],
    "Percentage": [df[skill].sum() / len(df) * 100 for skill in skill_cols],
}).sort_values("Job_Count", ascending=False)

skills_path = os.path.join(output_dir, "skills_summary.csv")
skills_summary.to_csv(skills_path, index=False)
print(f"   ✅ Saved: {skills_path}")

# 6) City summary
print("\n5️⃣ Creating city summary...")

city_summary = df.groupby("location_clean").agg({
    "job_id": "count",
    "salary_avg": ["mean", "median"],
    "company": "nunique",
}).round(0)

city_summary.columns = ["Job_Count", "Avg_Salary", "Median_Salary", "Company_Count"]
city_summary = city_summary.reset_index().sort_values("Job_Count", ascending=False)

city_path = os.path.join(output_dir, "city_summary.csv")
city_summary.to_csv(city_path, index=False)
print(f"   ✅ Saved: {city_path}")

# 7) Experience summary
print("\n6️⃣ Creating experience summary...")

experience_summary = df.groupby("experience_level").agg({
    "job_id": "count",
    "salary_avg": ["mean", "min", "max"],
}).round(0)

experience_summary.columns = ["Job_Count", "Avg_Salary", "Min_Salary", "Max_Salary"]
experience_summary = experience_summary.reset_index().sort_values("Avg_Salary", ascending=False)

exp_path = os.path.join(output_dir, "experience_summary.csv")
experience_summary.to_csv(exp_path, index=False)
print(f"   ✅ Saved: {exp_path}")

# 8) Job type summary
print("\n7️⃣ Creating job type summary...")

job_type_summary = df["job_type"].value_counts().reset_index()
job_type_summary.columns = ["Job_Type", "Count"]
job_type_summary["Percentage"] = job_type_summary["Count"] / len(df) * 100

job_type_path = os.path.join(output_dir, "job_type_summary.csv")
job_type_summary.to_csv(job_type_path, index=False)
print(f"   ✅ Saved: {job_type_path}")

# 9) Main dataset for Power BI
print("\n8️⃣ Exporting main dataset for Power BI...")

main_path = os.path.join(output_dir, "job_market_for_powerbi.csv")
df.to_csv(main_path, index=False)
print(f"   ✅ Saved: {main_path}  ({len(df)} rows)")

# 10) Monthly trend (if posted_date is available)
print("\n9️⃣ Creating monthly trend file...")

if "posted_date" in df.columns:
    df_valid = df[df["posted_date"].notna()].copy()
    if len(df_valid) > 0:
        df_valid["month_year"] = df_valid["posted_date"].dt.to_period("M").astype(str)
        monthly_trend = df_valid.groupby("month_year").agg({
            "job_id": "count",
            "salary_avg": "mean",
        }).round(0)
        monthly_trend.columns = ["Job_Count", "Avg_Salary"]
        monthly_trend = monthly_trend.reset_index().sort_values("month_year")

        month_path = os.path.join(output_dir, "monthly_trend.csv")
        monthly_trend.to_csv(month_path, index=False)
        print(f"   ✅ Saved: {month_path}")
    else:
        print("   ⚠️ No valid dates for monthly trend; skipping.")
else:
    print("   ⚠️ 'posted_date' not available; monthly trend file not created.")

print("\n" + "=" * 60)
print("✅ EXPORT FOR POWER BI COMPLETE!")
print("=" * 60)

print("\n📁 Files created in output folder:")
for f in os.listdir(output_dir):
    print("   •", f)

📤 EXPORTING DATA FOR POWER BI

📂 Project root: C:\Users\yelle\Job_Market_Analytics
📂 Cleaned data folder: C:\Users\yelle\Job_Market_Analytics\data\cleaned
📂 Output folder: C:\Users\yelle\Job_Market_Analytics\output

1️⃣ Loading cleaned data...
   ✅ Loaded 376 rows
   📊 Columns: ['job_id', 'job_title', 'company', 'location', 'salary_min', 'salary_max', 'job_type', 'experience_level', 'company_size', 'required_skills', 'posted_date', 'company_description', 'salary_avg', 'job_title_clean', 'location_clean', 'month', 'day_of_week', 'Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Machine Learning', 'TensorFlow', 'PyTorch', 'AWS', 'Azure', 'Google Cloud', 'R', 'JavaScript', 'Docker', 'Kubernetes', 'Airflow', 'Spark', 'Hadoop', 'DAX', 'Visualization', 'Git', 'Jupyter', 'Pandas', 'NumPy', 'total_skills']

2️⃣ Converting posted_date to datetime...
   ✅ Date conversion done.

3️⃣ Creating overall metrics...
   ✅ Saved: C:\Users\yelle\Job_Market_Analytics\output\overall_metrics.csv

4️⃣ Creatin